# Concert Revenue PredictorA single-stage model that predicts concert revenue directly from artist, venue, price, market, and time inputs.**Why a fresh build:** the two-stage model accumulated a lot of architectural debt over time (LEAKAGE_VARS conflicts between Stage 1 and Stage 2, model file mismatches, contradictory feature handling). This is a focused companion model for the "predict revenue for a real concert" use case Justin asked about.**What it does:**1. Predicts revenue for a given concert configuration (artist + venue + price + date + market)2. Recommends the best price × venue capacity combination from a search grid3. Returns a confidence range alongside each prediction**Target:** `avg_gross_usd` (z-scored in the dataset; real-dollar conversion available if scaler stats provided)**Model:** XGBoost gradient boosting, trained with GroupKFold cross-validation by headliner to avoid leakage.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
np.random.seed(42)

## Load Data

In [ ]:
# Update this path to wherever your CSV lives
DATA_PATH = 'test_sample_cleaned_v2.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} events, {df.shape[1]} columns')
print(f'Unique headliners: {df["headliner"].nunique()}')

# Try to load scaler stats for real-dollar output
# If you have scaler_stats_apr8.json from the scoring engine project, place it next to this notebook
SCALER_PATH = 'scaler_stats_apr8.json'
scaler_stats = None
if Path(SCALER_PATH).exists():
    with open(SCALER_PATH) as f:
        scaler_stats = json.load(f)
    print(f'Scaler stats loaded - predictions will show real dollar values')
else:
    print(f'No scaler stats found at {SCALER_PATH} - predictions will be in z-score units')
    print('(To get dollar values, copy scaler_stats_apr8.json next to this notebook)')

def to_real(scaled_val, col):
    """Inverse z-score: scaled --> original units (works only if scaler_stats loaded)"""
    if scaler_stats is None or col not in scaler_stats:
        return scaled_val
    return scaled_val * scaler_stats[col]['std'] + scaler_stats[col]['mean']

def to_scaled(real_val, col):
    """Z-score: original units --> scaled"""
    if scaler_stats is None or col not in scaler_stats:
        return real_val
    return (real_val - scaler_stats[col]['mean']) / scaler_stats[col]['std']

def fmt_revenue(z_val):
    """Format a z-scored revenue value as a dollar string when possible"""
    if scaler_stats and 'avg_gross_usd' in scaler_stats:
        return f'${to_real(z_val, "avg_gross_usd"):,.0f}'
    return f'{z_val:+.3f} (z-score)'

## Feature Selection**Target:** `avg_gross_usd`**Excluded as outcome variables (would cause leakage):**- `avg_tickets_sold` (component of revenue)- `avg_capacity_sold` (fill rate, computed from tickets and capacity)- `ticket_price_min`, `ticket_price_max` (post-hoc summaries)- `number_of_shows` (event count is part of the aggregation)**Excluded as non-features (identifiers/redundant text):**- IDs, dates (we have month/day/year separately), free-text fields- The one-hot encoded versions of these are kept**Kept (this is what the model sees):**- `ticket_price_avg`, `avg_event_capacity` — controllable inputs- Artist popularity signals (Google Trends, Wikipedia, historical concerts)- Market features (population, income)- Time features (month_sin/cos, day_of_week_sin/cos, year_offset, lockdown flag)- One-hot genre, city, state, market, company_type, promoter, headliner

In [ ]:
TARGET = 'avg_gross_usd'

# Variables derived from the target itself - dropping prevents leakage
LEAKAGE = [
    'avg_tickets_sold',
    'avg_capacity_sold',
    'ticket_price_min',
    'ticket_price_max',
    'number_of_shows',
]

# Raw text/ID columns - the cleaned one-hot versions are kept
NON_FEATURES = [
    'eventid', 'event_date',
    'headliner', 'support', 'venue', 'city', 'state', 'country', 'market',
    'company_type', 'promoter', 'genre',
    'gt_date_range', 'last_album_date',
    'census_market_name', 'market_clean', 'acs_market',
    'day_of_week', 'month',  # keep sin/cos versions instead
    'is_missing_album_dates', 'is_missing_support',
]

# Group key for cross-validation (do this BEFORE dropping headliner)
groups = df['headliner'].fillna('UNKNOWN').values

drop_cols = [c for c in (LEAKAGE + NON_FEATURES) if c in df.columns]
X = df.drop(columns=drop_cols + [TARGET], errors='ignore')
y = df[TARGET]

# Convert any boolean columns to int (XGBoost prefers numeric)
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

# Drop any remaining non-numeric columns (XGBoost can't handle strings)
non_numeric = X.select_dtypes(exclude='number').columns.tolist()
if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    X = X.drop(columns=non_numeric)

print(f'Final feature count: {X.shape[1]}')
print(f'Target: {TARGET}')
print(f'Group key for CV: headliner ({len(set(groups))} unique groups)')

## Cross-Validated TrainingUse **GroupKFold by headliner** so the same artist never appears in both train and test. Without this, the model can memorize artist-specific patterns and inflate its test scores — a real risk given that 5 artists account for 25% of the events in this dataset.We train one model per fold, average the scores, then train the final model on all data.

In [ ]:
# 5-fold GroupKFold cross-validation
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

# Standard XGBoost config - mild regularization to combat the right-skewed target
XGB_PARAMS = dict(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=3.0,
    min_child_weight=5,
    random_state=42,
    n_jobs=-1,
)

fold_scores = []
for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    m = XGBRegressor(**XGB_PARAMS)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    
    fold_scores.append({
        'fold': fold_idx,
        'train_n': len(train_idx),
        'test_n': len(test_idx),
        'MAE': mean_absolute_error(y_te, pred),
        'RMSE': np.sqrt(mean_squared_error(y_te, pred)),
        'R2': r2_score(y_te, pred),
    })

cv_df = pd.DataFrame(fold_scores)
print('Cross-validation results (GroupKFold by headliner):')
print(cv_df.to_string(index=False))
print(f'\nMean test MAE:  {cv_df["MAE"].mean():.4f} (+/- {cv_df["MAE"].std():.4f})')
print(f'Mean test RMSE: {cv_df["RMSE"].mean():.4f} (+/- {cv_df["RMSE"].std():.4f})')
print(f'Mean test R^2:  {cv_df["R2"].mean():.4f} (+/- {cv_df["R2"].std():.4f})')

## Final Model: Trained on All DataNow that cross-validation has given us an honest performance estimate, train the production model on the full dataset.

In [ ]:
model = XGBRegressor(**XGB_PARAMS)
model.fit(X, y)

# Sanity check: training fit (will be optimistic compared to CV)
y_pred_train = model.predict(X)
print(f'Training fit (optimistic - use CV scores above for real expectations):')
print(f'  MAE:  {mean_absolute_error(y, y_pred_train):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y, y_pred_train)):.4f}')
print(f'  R^2:  {r2_score(y, y_pred_train):.4f}')

# Save the model and feature list for downstream use
model.save_model('revenue_predictor_v1.json')
with open('revenue_predictor_features.json', 'w') as f:
    json.dump(list(X.columns), f)
print(f'\nSaved: revenue_predictor_v1.json')
print(f'Saved: revenue_predictor_features.json')

## Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

top20 = importances.head(20)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20.index[::-1], top20.values[::-1], color='#2D8B4E', edgecolor='white')
ax.set_xlabel('Feature importance')
ax.set_title('Top 20 Features - Revenue Predictor', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
for feat, val in importances.head(10).items():
    print(f'  {feat:<35} {val:.4f}')

## Predict Revenue for a Single ConcertThe `predict_revenue()` function takes a row of feature values and returns a predicted revenue plus a rough confidence range. The confidence range is derived from the cross-validation RMSE — it's an honest estimate of how wrong we'd typically expect to be.

In [ ]:
# Use CV RMSE as the basis for confidence intervals
CV_RMSE = cv_df['RMSE'].mean()

def predict_revenue(feature_row, model=model, feature_list=list(X.columns)):
    """
    Predict revenue for a single concert configuration.
    
    Parameters
    ----------
    feature_row : dict or pd.Series
        Feature values for the concert. Any missing features default to dataset median.
    
    Returns
    -------
    dict with keys: 'predicted_z', 'predicted_dollars', 'lo_dollars', 'hi_dollars'
    """
    # Start with median values from training data for any missing features
    full_row = X.median(numeric_only=True).copy()
    if isinstance(feature_row, dict):
        feature_row = pd.Series(feature_row)
    for col, val in feature_row.items():
        if col in full_row.index:
            full_row[col] = val
    
    # Predict
    X_input = pd.DataFrame([full_row])[feature_list]
    pred_z = float(model.predict(X_input)[0])
    
    # 1-sigma confidence band from CV RMSE
    lo_z = pred_z - CV_RMSE
    hi_z = pred_z + CV_RMSE
    
    return {
        'predicted_z':       pred_z,
        'predicted_dollars': fmt_revenue(pred_z),
        'lo_dollars':        fmt_revenue(lo_z),
        'hi_dollars':        fmt_revenue(hi_z),
    }

# Test on event index 58 (Portugal. The Man, the team's running test case)
test_idx = 58
test_features = X.iloc[test_idx].to_dict()
test_result = predict_revenue(test_features)

actual_rev_z = y.iloc[test_idx]
print(f'Test prediction for event {test_idx} ({df.iloc[test_idx]["headliner"]}):')
print(f'  Predicted:   {test_result["predicted_dollars"]}')
print(f'  Range:       {test_result["lo_dollars"]} -- {test_result["hi_dollars"]}')
print(f'  Actual:      {fmt_revenue(actual_rev_z)}')

## Recommend the Best Price x Venue CombinationThe `recommend_best_combo()` function takes an artist profile and searches over a grid of price levels and venue capacities to find the combination with the highest predicted revenue. This is the "what should we charge and where should we book?" question.The search range for prices and capacities defaults to peer-group percentiles (similar to the two-stage scoring engine's approach), but unlike the two-stage model, this is a single predictor doing direct revenue optimization.

In [ ]:
def recommend_best_combo(feature_row, grid_size=5, peer_filter=None, model=model,
                          feature_list=list(X.columns), source_df=df):
    """
    Search over price x capacity combinations and return the best one.
    
    Parameters
    ----------
    feature_row : dict or pd.Series
        Base feature values for the concert (everything except price and capacity).
    grid_size : int
        How many price levels and capacity levels to test (grid_size x grid_size combinations).
    peer_filter : dict, optional
        Restrict the peer group used to set the search range, e.g. {'genre_cleaned_pop_rock': 1}.
        If None, uses the full dataset for price/capacity percentiles.
    
    Returns
    -------
    dict with the best combo plus a full grid of predictions.
    """
    # Determine the peer group for price/capacity ranges
    if peer_filter:
        mask = pd.Series([True] * len(source_df))
        for col, val in peer_filter.items():
            if col in source_df.columns:
                mask &= (source_df[col] == val)
        peers = source_df[mask]
        if len(peers) < 20:  # fallback if peer group too narrow
            peers = source_df
    else:
        peers = source_df
    
    # Build the search grid from peer percentiles
    pct_lo, pct_hi = 0.10, 0.90
    percentiles = np.linspace(pct_lo, pct_hi, grid_size)
    price_levels = peers['ticket_price_avg'].quantile(percentiles).values
    cap_levels   = peers['avg_event_capacity'].quantile(percentiles).values
    
    # Base row with median defaults
    base = X.median(numeric_only=True).copy()
    if isinstance(feature_row, dict):
        feature_row = pd.Series(feature_row)
    for col, val in feature_row.items():
        if col in base.index:
            base[col] = val
    
    # Score every cell in the grid
    grid = np.zeros((grid_size, grid_size))
    best_rev, best_pi, best_ci = -np.inf, None, None
    
    for ci, cap_z in enumerate(cap_levels):
        for pi, price_z in enumerate(price_levels):
            row = base.copy()
            row['ticket_price_avg']    = price_z
            row['avg_event_capacity']  = cap_z
            X_input = pd.DataFrame([row])[feature_list]
            pred_z = float(model.predict(X_input)[0])
            grid[ci, pi] = pred_z
            if pred_z > best_rev:
                best_rev, best_pi, best_ci = pred_z, pi, ci
    
    return {
        'best_price_z':       price_levels[best_pi],
        'best_capacity_z':    cap_levels[best_ci],
        'best_price_real':    to_real(price_levels[best_pi], 'ticket_price_avg'),
        'best_capacity_real': to_real(cap_levels[best_ci], 'avg_event_capacity'),
        'best_revenue_dollars': fmt_revenue(best_rev),
        'best_revenue_z':     best_rev,
        'grid_z':             grid,
        'price_levels_z':     price_levels,
        'cap_levels_z':       cap_levels,
        'price_levels_real':  [to_real(p, 'ticket_price_avg') for p in price_levels],
        'cap_levels_real':    [to_real(c, 'avg_event_capacity') for c in cap_levels],
    }

# Test on event 58 again
rec = recommend_best_combo(test_features)
print(f'Recommended for event {test_idx} ({df.iloc[test_idx]["headliner"]}):')
print(f'  Best price:    ${rec["best_price_real"]:,.0f} (z={rec["best_price_z"]:+.2f})' if scaler_stats else f'  Best price (z): {rec["best_price_z"]:+.2f}')
print(f'  Best capacity: {rec["best_capacity_real"]:,.0f} seats' if scaler_stats else f'  Best capacity (z): {rec["best_capacity_z"]:+.2f}')
print(f'  Expected revenue: {rec["best_revenue_dollars"]}')

## Visualize the Recommendation Grid

In [ ]:
def plot_grid(rec, title='Revenue by Price x Capacity'):
    fig, ax = plt.subplots(figsize=(9, 6))
    grid = rec['grid_z']
    im = ax.imshow(grid, cmap='YlGn', aspect='auto')
    
    best_ci = np.argmax(grid) // grid.shape[1]
    best_pi = np.argmax(grid) % grid.shape[1]
    
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            val = grid[i, j]
            txt = fmt_revenue(val) if scaler_stats else f'{val:+.2f}'
            if (i, j) == (best_ci, best_pi):
                txt = '★\n' + txt
            ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                    fontweight='bold' if (i, j) == (best_ci, best_pi) else 'normal')
    
    if scaler_stats:
        x_labels = [f'${p:,.0f}' for p in rec['price_levels_real']]
        y_labels = [f'{c:,.0f}' for c in rec['cap_levels_real']]
    else:
        x_labels = [f'{p:+.2f}' for p in rec['price_levels_z']]
        y_labels = [f'{c:+.2f}' for c in rec['cap_levels_z']]
    
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel('Ticket Price')
    ax.set_ylabel('Venue Capacity')
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Predicted Revenue')
    plt.tight_layout()
    plt.show()

plot_grid(rec, title=f'Revenue Grid: {df.iloc[test_idx]["headliner"]} (event {test_idx})')

## Demo: 3 Real-Style ConcertsThree example concert configurations to show how the model behaves across genres.

In [ ]:
demo_events = []

# Pop/Rock event - take a real event and use its features
pop_idx = df[df['genre_cleaned_pop_rock'] == 1].index[5]
demo_events.append(('Pop/Rock example', X.iloc[pop_idx].to_dict(), df.iloc[pop_idx]['headliner']))

# Country event
country_idx = df[df['genre_cleaned_country'] == 1].index[5]
demo_events.append(('Country example', X.iloc[country_idx].to_dict(), df.iloc[country_idx]['headliner']))

# Latin event
latin_idx = df[df['genre_cleaned_latin'] == 1].index[5]
demo_events.append(('Latin example', X.iloc[latin_idx].to_dict(), df.iloc[latin_idx]['headliner']))

print(f'{"Configuration":<22}{"Artist":<25}{"Predicted":>15}{"Recommended":>30}')
print('-' * 92)
for label, features, artist in demo_events:
    pred = predict_revenue(features)
    rec  = recommend_best_combo(features)
    artist_str = artist[:22] if isinstance(artist, str) else 'Unknown'
    if scaler_stats:
        rec_str = f'${rec["best_price_real"]:.0f} x {rec["best_capacity_real"]:,.0f}seats'
    else:
        rec_str = f'price z={rec["best_price_z"]:+.2f}, cap z={rec["best_capacity_z"]:+.2f}'
    print(f'{label:<22}{artist_str:<25}{pred["predicted_dollars"]:>15}{rec_str:>30}')

## Summary**What this model is:**- A single-stage gradient boosting regressor that predicts concert revenue directly- Cross-validated with GroupKFold by headliner (no artist appears in both train and test)- Honest about uncertainty: every prediction comes with a ±1σ range based on CV error**What it does well:**- Single-pitch explanation: "model takes concert details, returns predicted revenue"- No two-stage architecture to maintain or debug- Cleaner feature handling - everything is in one place- Includes `recommend_best_combo()` for price/venue optimization**Limitations to flag for the team:**- Same dataset limitations apply (1,808 events, 208 artists, heavy Pollstar bias)- Does NOT replace the two-stage model for elasticity-based pricing decisions - this directly optimizes revenue without modeling elasticity- The recommendation may still favor higher prices because the data has the same endogeneity (promoters set high prices for popular shows)- No causal interpretation - this is correlational**Files saved:**- `revenue_predictor_v1.json` - the trained model- `revenue_predictor_features.json` - the feature list- `feature_importance.png` - top 20 features plot